In [2]:
import torch
import torch.nn as nn

In [3]:
# 5개의 입력과 6개의 출력을 가진 신경망 층에 2개의 입력 샘플을 적용

torch.manual_seed(123)

batch_example = torch.randn(2, 5)

layer = nn.Sequential(
    nn.Linear(5, 6),
    nn.ReLU()
)

out = layer(batch_example)
print(f"out:\n{out}\n")

# 층 정규화를 적용하기 전에 평균과 분산을 확인
mean = out.mean(dim=-1, keepdim=True)
mean_test = out.mean(dim=-1)
var = out.var(dim=-1, keepdim=True)
var_test = out.var(dim=-1)
print(f"평균: {mean}")
print(f"분산: {var}\n")

# 앞서 얻은 층 출력에 층 정규화를 적용
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print(f"{out_norm}")
print(f"평균: {mean}")
print(f"분산: {var}\n")

# 가독성을 위해 sci_mode를 False로 지정하여 텐서를 출력할 때 과학적 표기법을 사용하지 않을 수도 있다.
torch.set_printoptions(sci_mode=False)
print(f"평균: {mean}")
print(f"분산: {var}")


out:
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)

평균: tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
분산: tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)

tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
평균: tensor([[9.9341e-09],
        [0.0000e+00]], grad_fn=<MeanBackward1>)
분산: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)

평균: tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
분산: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [4]:
# GPT-2 모델 설정값
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # 어휘사전 크기
    "context_length": 1024, # 문맥 길이
    "emb_dim": 768,         # 임베딩 차원
    "n_heads": 12,          # 어텐션 헤드 개수
    "n_layers": 12,         # 층 개수
    "drop_rate": 0.1,       # 드롭아웃 비율
    "qkv_bias": False       # 쿼리, 키, 값 계산을 위한 편향
}

In [5]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(num_embeddings=cfg["vocab_size"], embedding_dim=cfg["emb_dim"])
        self.pos_emb = nn.Embedding(num_embeddings=cfg["context_length"], embedding_dim=cfg["emb_dim"])
        self.drop_emb = nn.Dropout(p=cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(in_features=cfg["emb_dim"], out_features=cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len), device=in_idx.device)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, enps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [6]:
# 코드 4-2 층 정규화 클래스

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [7]:
# 이제 LayerNorm 모듈을 배치에 적용

ln = LayerNorm(emb_dim=5)

print(f"Before Norm:\n{batch_example}\n")
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, keepdim=True)

print(f"After Norm:\n{out_ln}\n")
print(f"평균: {mean}")
print(f"분산: {var}")

Before Norm:
tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

After Norm:
tensor([[ 0.5528,  1.0693, -0.0223,  0.2656, -1.8654],
        [ 0.9087, -1.3767, -0.9564,  1.1304,  0.2940]], grad_fn=<AddBackward0>)

평균: tensor([[-0.0000],
        [ 0.0000]], grad_fn=<MeanBackward1>)
분산: tensor([[1.2499],
        [1.2500]], grad_fn=<VarBackward0>)
